In [ ]:
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML
import numpy as np
import pandas as pd
import sys 
sys.path.insert(
    0,
    "/raid0/homes/jkitzmann/dev/ITK-build-binarythinning/Wrapping/Generators/Python"
)
import itk
from scipy.ndimage import map_coordinates
from scipy import ndimage
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist

In [ ]:
PixelType = itk.UC
Dimension = 3
ImageType = itk.Image[PixelType, Dimension]
FloatImageType = itk.Image[itk.F, Dimension]

def _to_xyz_list(points):
    """Normalize show_points into a list of (x, y, z) tuples.

    Accepts a single ITK index (or (x, y, z) sequence), or a list of them.
    """
    if points is None:
        return []

    # a bare (x, y, z) is one point; a list/tuple of those is many
    if not (isinstance(points, (list, tuple)) and not np.isscalar(points[0])):
        points = [points]

    return [(int(p[0]), int(p[1]), int(p[2])) for p in points]

def show_mip(image, show_points=None, cmap="gray"):
    ar = itk.array_view_from_image(image)

    points = _to_xyz_list(show_points)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(np.max(ar, axis=0), cmap=cmap)
    axes[0].set_title("Axial MIP")
    axes[0].axis("off")

    axes[1].imshow(np.max(ar, axis=1), cmap=cmap)
    axes[1].set_title("Coronal MIP")
    axes[1].axis("off")

    axes[2].imshow(np.max(ar, axis=2), cmap=cmap)
    axes[2].set_title("Sagittal MIP")
    axes[2].axis("off")

    # add points from list
    for ax, (col, row) in zip(axes, [(0, 1), (0, 2), (1, 2)]):
        ax.autoscale(False)
        for point in points:
            ax.scatter(
                point[col],
                point[row],
                s=80,
                facecolors="none",
                edgecolors="red",
                linewidths=1.5,
            )

    plt.tight_layout()
    plt.show()

In [ ]:
!pwd

In [ ]:
fixed_image = itk.imread(r'../ZUNU_vida-airtree.hdr', PixelType)
# fixed_image = itk.imread(r'../RUL_simple_512.nii', PixelType)

show_mip(fixed_image)

## Deform image to create moving comparison

In [ ]:
WAVE_AMPLITUDE = 2.0 
WAVE_CYCLES = 3
WAVE_PHASE = 0.0

TRANSLATION_MM = np.array([6.0, -4.0, 3.0])

ORIGIN_SHIFT_MM = np.array([10.0, 5.0, -5.0])

size = fixed_image.GetLargestPossibleRegion().GetSize()
spacing = np.array(fixed_image.GetSpacing())
shape = tuple(reversed(size)) 

nz, ny, nx = shape

s_extent_mm = (nz - 1) * spacing[2]
s_coord_mm = np.arange(nz) * spacing[2]

wave_1d = WAVE_AMPLITUDE * np.sin(
    2.0 * np.pi * WAVE_CYCLES * s_coord_mm / s_extent_mm + WAVE_PHASE
)

l = np.broadcast_to(wave_1d[:, None, None], shape).astype(np.float32).copy()
p = np.zeros(shape, dtype=np.float32)
s = np.zeros(shape, dtype=np.float32)

l -= TRANSLATION_MM[0]
p -= TRANSLATION_MM[1]
s -= TRANSLATION_MM[2]

magnitude = np.sqrt(l**2 + p**2 + s**2)
MAX_DISPLACEMENT = float(magnitude.max())

print(f"Max displacement:    {magnitude.max():.3f} mm")
print(f"Mean displacement:   {magnitude.mean():.3f} mm")
print(f"Median displacement: {np.median(magnitude):.3f} mm")
print(f"95th percentile:     {np.percentile(magnitude, 95):.3f} mm")

l_image = itk.image_from_array(l.astype(np.float32))
p_image = itk.image_from_array(p.astype(np.float32))
s_image = itk.image_from_array(s.astype(np.float32))

for image in (l_image, p_image, s_image):
    image.CopyInformation(fixed_image)

VectorPixelType = itk.Vector[itk.F, 3]

DisplacementFieldType = itk.Image[
    VectorPixelType,
    3
]

compose = itk.ComposeImageFilter[
    FloatImageType,
    DisplacementFieldType
].New()

compose.SetInput(0, l_image)
compose.SetInput(1, p_image)
compose.SetInput(2, s_image)

compose.Update()

displacement_field = compose.GetOutput()

warpFilter = itk.WarpImageFilter[
    ImageType,
    ImageType,
    DisplacementFieldType
].New()

warpFilter.SetInterpolator(
    itk.NearestNeighborInterpolateImageFunction[
        ImageType,
        itk.D
    ].New()
)

warpFilter.SetOutputSpacing(fixed_image.GetSpacing())
warpFilter.SetOutputOrigin(fixed_image.GetOrigin())
warpFilter.SetOutputDirection(fixed_image.GetDirection())

warpFilter.SetDisplacementField(displacement_field)
warpFilter.SetInput(fixed_image)

warpFilter.Update()

moving_image = warpFilter.GetOutput()
moving_image.DisconnectPipeline()
moving_image.SetOrigin((np.array(fixed_image.GetOrigin()) + ORIGIN_SHIFT_MM).tolist())

print("Fixed: ", itk.size(fixed_image), list(fixed_image.GetOrigin()))
print("Moving:", itk.size(moving_image), list(moving_image.GetOrigin()))
print("Rigid offset to recover:", list(TRANSLATION_MM + ORIGIN_SHIFT_MM), "mm (+ local wave)")

In [ ]:
show_mip(moving_image)

## Preprocessing

In [ ]:
CropToMaskRegionType = itk.CropToMaskRegion[ImageType]

cropToMaskRegionPrediction = CropToMaskRegionType.New()
cropToMaskRegionPrediction.SetInput(moving_image)
cropToMaskRegionPrediction.Update()
moving_image_cropped = cropToMaskRegionPrediction.GetOutput()
show_mip(moving_image_cropped)

cropToMaskRegionGroundTruth = CropToMaskRegionType.New()
cropToMaskRegionGroundTruth.SetInput(fixed_image)
cropToMaskRegionGroundTruth.Update()
fixed_image_cropped = cropToMaskRegionGroundTruth.GetOutput()
show_mip(fixed_image_cropped)


# -- Nonrigid Pipeline --

## Skeletonization

In [ ]:
SkeletonizeAirwayType = itk.SkeletonizeAirway[ImageType]

skeletonizeAirwayPrediction = SkeletonizeAirwayType.New()
skeletonizeAirwayPrediction.SetInput(moving_image_cropped)
skeletonizeAirwayPrediction.Update()
moving_skeleton = skeletonizeAirwayPrediction.GetOutput()
show_mip(moving_skeleton)

skeletonizeAirwayGroundTruth = SkeletonizeAirwayType.New()
skeletonizeAirwayGroundTruth.SetInput(fixed_image_cropped)
skeletonizeAirwayGroundTruth.Update()
fixed_skeleton = skeletonizeAirwayGroundTruth.GetOutput()
show_mip(fixed_skeleton)

## Label Both Images

In [ ]:
LabelPixelType = itk.SS
LabelImageType = itk.Image[LabelPixelType, Dimension]

SkeletonLabelingType = itk.SkeletonLabeling[ImageType, LabelImageType]

skeletonLabelingPrediction = SkeletonLabelingType.New()
skeletonLabelingPrediction.SetMaskImage(moving_image_cropped)
skeletonLabelingPrediction.SetSkeletonImage(moving_skeleton)
skeletonLabelingPrediction.SetSensitivityMultiplier(1.0)
skeletonLabelingPrediction.Update()
branchLabelsImagePrediction = skeletonLabelingPrediction.GetBranchLabelsImage()
generationLabelsImagePrediction = skeletonLabelingPrediction.GetGenerationLabelsImage()
distanceMapImagePrediction = skeletonLabelingPrediction.GetDistanceMapImage()
widestIndexPrediction = skeletonLabelingPrediction.GetWidestIndex()

print("widest index:", list(widestIndexPrediction))
show_mip(distanceMapImagePrediction, show_points=widestIndexPrediction)

skeletonLabelingGroundTruth = SkeletonLabelingType.New()
skeletonLabelingGroundTruth.SetMaskImage(fixed_image_cropped)
skeletonLabelingGroundTruth.SetSkeletonImage(fixed_skeleton)
skeletonLabelingGroundTruth.SetSensitivityMultiplier(1.0)
skeletonLabelingGroundTruth.Update()
branchLabelsImageGroundTruth = skeletonLabelingGroundTruth.GetBranchLabelsImage()
generationLabelsImageGroundTruth = skeletonLabelingGroundTruth.GetGenerationLabelsImage()
distanceMapImageGroundTruth = skeletonLabelingGroundTruth.GetDistanceMapImage()
widestIndexGroundTruth = skeletonLabelingGroundTruth.GetWidestIndex()

print("widest index:", list(widestIndexGroundTruth))
show_mip(distanceMapImageGroundTruth, show_points=widestIndexGroundTruth)

In [ ]:
branch_labels_prediction = itk.array_view_from_image(branchLabelsImagePrediction)
show_mip(branchLabelsImagePrediction, show_points=widestIndexPrediction, cmap="nipy_spectral")

branch_labels_groundTruth = itk.array_view_from_image(branchLabelsImageGroundTruth)
show_mip(branchLabelsImageGroundTruth, show_points=widestIndexGroundTruth, cmap="nipy_spectral")


In [ ]:
ProjectAirwayType = itk.ProjectAirway[ImageType, LabelImageType]

projectAirwayPrediction = ProjectAirwayType.New()
projectAirwayPrediction.SetSegmentationImage(moving_image_cropped)
projectAirwayPrediction.SetBranchSkeletonLabelsImage(branchLabelsImagePrediction)
projectAirwayPrediction.SetGenerationSkeletonLabelsImage(generationLabelsImagePrediction)
projectAirwayPrediction.SetNumberOfNeighbors(5)
projectAirwayPrediction.Update()
projectedBranchLabelsImagePrediction = projectAirwayPrediction.GetBranchLabelsImage()
show_mip(projectedBranchLabelsImagePrediction, cmap="nipy_spectral")

projectAirwayGroundTruth = ProjectAirwayType.New()
projectAirwayGroundTruth.SetSegmentationImage(fixed_image_cropped)
projectAirwayGroundTruth.SetBranchSkeletonLabelsImage(branchLabelsImageGroundTruth)
projectAirwayGroundTruth.SetGenerationSkeletonLabelsImage(generationLabelsImageGroundTruth)
projectAirwayGroundTruth.SetNumberOfNeighbors(5)
projectAirwayGroundTruth.Update()
projectedBranchLabelsImageGroundTruth = projectAirwayGroundTruth.GetBranchLabelsImage()
show_mip(projectedBranchLabelsImageGroundTruth, cmap="nipy_spectral")

moving_image = projectedBranchLabelsImagePrediction
fixed_image = projectedBranchLabelsImageGroundTruth

print(type(moving_image))
print(type(fixed_image))


## Align Images by Trachea

In [ ]:
n_branches_prediction = skeletonLabelingPrediction.GetNumberOfBranches()
branch_ids_prediction = range(1, n_branches_prediction + 1)
branch_origins_prediction = [skeletonLabelingPrediction.GetBranchOriginIndex(i) for i in branch_ids_prediction]

n_branches_groundTruth = skeletonLabelingGroundTruth.GetNumberOfBranches()
branch_ids_groundTruth = range(1, n_branches_groundTruth + 1)
branch_origins_groundTruth = [skeletonLabelingGroundTruth.GetBranchOriginIndex(i) for i in branch_ids_groundTruth]

show_mip(moving_image, branch_origins_prediction)
show_mip(fixed_image, branch_origins_groundTruth)

In [ ]:
# trachea will be the second branch point in the heirarchy (first is the "throat")
trachea_fixed = branch_origins_groundTruth[1]
trachea_moving = branch_origins_prediction[1]

print('fixed image trachea index: ', trachea_fixed)
print('moving image trachea index: ', trachea_moving)

trachea_point_fixed = projectedBranchLabelsImageGroundTruth.TransformIndexToPhysicalPoint(trachea_fixed)
trachea_point_moving= projectedBranchLabelsImagePrediction.TransformIndexToPhysicalPoint(trachea_moving)

print('fixed image trachea point: ', trachea_point_fixed)
print('moving image trachea point: ', trachea_point_moving)

show_mip(projectedBranchLabelsImagePrediction, trachea_moving)
show_mip(projectedBranchLabelsImageGroundTruth, trachea_fixed)


In [ ]:
displacement = trachea_point_moving - trachea_point_fixed
print(displacement)

transform = itk.TranslationTransform[itk.D, Dimension].New()
transform.SetOffset(displacement)

interpolator = itk.NearestNeighborInterpolateImageFunction.New(moving_image)

resampled = itk.resample_image_filter(
    moving_image,
    transform=transform,
    interpolator=interpolator,
    use_reference_image=True,
    reference_image=fixed_image,
)

In [ ]:
show_mip(resampled)

In [ ]:
def _overlay_channel(array, binarize):
    if binarize:
        return (array != 0).astype(np.float32)

    array = array.astype(np.float32)
    lo, hi = float(array.min()), float(array.max())
    return np.zeros_like(array) if hi <= lo else (array - lo) / (hi - lo)


# show two MIPs together on different color chanels to compare alignment
def show_overlay(image_red, image_blue, labels=("red", "blue"),
                 binarize=True):
    ar_red = itk.array_view_from_image(image_red)
    ar_blue = itk.array_view_from_image(image_blue)

    channel_red = _overlay_channel(ar_red, binarize)
    channel_blue = _overlay_channel(ar_blue, binarize)

    title = f"{labels[0]} = red, {labels[1]} = blue, agreement = white"

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    panels = zip(axes, (0, 1, 2), ("Axial MIP", "Coronal MIP", "Sagittal MIP"))

    for ax, axis, panel_title in panels:
        projected_red = channel_red.max(axis=axis)
        projected_blue = channel_blue.max(axis=axis)
        rgb = np.stack(
            [projected_red,
             np.minimum(projected_red, projected_blue),
             projected_blue],
            axis=-1,
        )
        ax.imshow(rgb)
        ax.set_title(panel_title)
        ax.axis("off")

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


In [ ]:
identity_transform = itk.TranslationTransform[itk.D, Dimension].New()
identity_transform.SetIdentity()
unaligned = itk.resample_image_filter(
    moving_image,
    transform=identity_transform,
    interpolator=interpolator,
    use_reference_image=True,
    reference_image=fixed_image,
)

show_overlay(fixed_image, unaligned, labels=("fixed", "moving (before align)"))
show_overlay(fixed_image, resampled, labels=("fixed", "moving (after align)"))

## Build Terminal and Nonterminal Images

In [ ]:
def _branch_generation(parent_of, branch_id):
    generation, visited = 0, set()

    while parent_of.get(branch_id, 0) != 0:
        if branch_id in visited:
            raise ValueError(f"cycle in branch parents at branch {branch_id}")
        visited.add(branch_id)
        branch_id = parent_of[branch_id]
        generation += 1

    return generation


def branch_table(labeling, image_name):
    branch_ids = range(1, labeling.GetNumberOfBranches() + 1)

    parent_of = {
        branch_id: int(labeling.GetBranchParentId(branch_id))
        for branch_id in branch_ids
    }

    rows = []
    for branch_id in branch_ids:
        index = labeling.GetBranchOriginIndex(branch_id)
        point = labeling.GetBranchOriginPoint(branch_id)

        rows.append({
            "image": image_name,
            "branch_id": branch_id,
            "is_terminal": bool(labeling.IsBranchTerminal(branch_id)),
            "parent_id": parent_of[branch_id],
            "generation": _branch_generation(parent_of, branch_id),
            "origin_i": int(index[0]),
            "origin_j": int(index[1]),
            "origin_k": int(index[2]),
            "origin_x": float(point[0]),
            "origin_y": float(point[1]),
            "origin_z": float(point[2]),
        })

    return pd.DataFrame(rows)


branches_prediction = branch_table(skeletonLabelingPrediction, "prediction")
branches_groundTruth = branch_table(skeletonLabelingGroundTruth, "ground truth")
branches = pd.concat([branches_prediction, branches_groundTruth], ignore_index=True)

print(branches.groupby(["image", "is_terminal"]).size().rename("branches"))
print()
print(branches.groupby(["image", "generation"]).size().rename("branches"))

branches

In [ ]:
def terminal_split(table):
    terminal = table.loc[table.is_terminal, "branch_id"].tolist()
    nonterminal = table.loc[~table.is_terminal, "branch_id"].tolist()
    return terminal, nonterminal


terminal_moving_labels, nonterminal_moving_labels = terminal_split(branches_prediction)
terminal_fixed_labels, nonterminal_fixed_labels = terminal_split(branches_groundTruth)

print('terminal prediction: ', terminal_moving_labels)
print('nonterminal prediction: ', nonterminal_moving_labels)
print()
print('terminal ground truth: ', terminal_fixed_labels)
print('nonterminal ground truth: ', nonterminal_fixed_labels)

In [ ]:
def mask_to_values(image, values):
    array = itk.array_view_from_image(image)
    values = [int(v) for v in values]

    missing = sorted(set(values) - set(np.unique(array).tolist()))
    if missing:
        print(f"warning: values absent from the image: {missing}")

    selected = np.where(np.isin(array, values), array, 0).astype(array.dtype, copy=False)

    result = itk.image_from_array(selected)
    result.CopyInformation(image)
    return result


terminal_moving = mask_to_values(resampled, terminal_moving_labels)
nonterminal_moving = mask_to_values(resampled, nonterminal_moving_labels)
terminal_fixed = mask_to_values(fixed_image, terminal_fixed_labels)
nonterminal_fixed = mask_to_values(fixed_image, nonterminal_fixed_labels)

show_mip(terminal_moving, cmap='nipy_spectral')
show_mip(nonterminal_moving, cmap='nipy_spectral')
show_mip(terminal_fixed, cmap='nipy_spectral')
show_mip(nonterminal_fixed, cmap='nipy_spectral')

In [ ]:
# drop any missing values from the scan 
def present_values(image):
    return set(np.unique(itk.array_view_from_image(image)).tolist()) - {0}


def drop_missing_values(values, present, name):
    kept = [v for v in values if v in present]
    dropped = [v for v in values if v not in present]

    if dropped:
        print(f"{name:>20}: dropped {len(dropped)} of {len(values)} -> {dropped}")
    else:
        print(f"{name:>20}: all {len(values)} present")

    return kept


present_moving = present_values(resampled)
present_fixed = present_values(fixed_image)

terminal_moving_labels = drop_missing_values(
    terminal_moving_labels, present_moving, "terminal moving")
nonterminal_moving_labels = drop_missing_values(
    nonterminal_moving_labels, present_moving, "nonterminal moving")

terminal_fixed_labels = drop_missing_values(
    terminal_fixed_labels, present_fixed, "terminal fixed")
nonterminal_fixed_labels = drop_missing_values(
    nonterminal_fixed_labels, present_fixed, "nonterminal fixed")

branches_prediction_present = branches_prediction[
    branches_prediction.branch_id.isin(terminal_moving_labels + nonterminal_moving_labels)
].reset_index(drop=True)

branches_groundTruth_present = branches_groundTruth[
    branches_groundTruth.branch_id.isin(terminal_fixed_labels + nonterminal_fixed_labels)
].reset_index(drop=True)

print()
print(f"prediction branches kept:   {len(branches_prediction_present)} of {len(branches_prediction)}")
print(f"ground truth branches kept: {len(branches_groundTruth_present)} of {len(branches_groundTruth)}")

In [ ]:
show_overlay(terminal_fixed, terminal_moving, labels=("terminal fixed", "terminal moving"))
show_overlay(nonterminal_fixed, nonterminal_moving, labels=("nonterminal fixed", "nonterminal moving"))

## Transform points to aligned space

In [ ]:
nonterminal_moving_points_df = branches_prediction[np.isin(branches_prediction["branch_id"], nonterminal_moving_labels)]
terminal_moving_points_df = branches_prediction[np.isin(branches_prediction["branch_id"], terminal_moving_labels)]

nonterminal_fixed_points_df = branches_groundTruth[np.isin(branches_groundTruth["branch_id"], nonterminal_fixed_labels)]
terminal_fixed_points_df = branches_groundTruth[np.isin(branches_groundTruth["branch_id"], terminal_fixed_labels)]

In [ ]:
def get_points(df):
    return np.array([
        df["origin_x"],
        df["origin_y"],
        df["origin_z"]
    ]).T

nonterminal_moving_points = get_points(nonterminal_moving_points_df)
terminal_moving_points = get_points(terminal_moving_points_df)

nonterminal_fixed_points = get_points(nonterminal_fixed_points_df)
terminal_fixed_points = get_points(terminal_fixed_points_df)

nonterminal_moving_points

In [ ]:
def transform_points(points, transform):
    return np.array([
        transform.TransformPoint(tuple(point))
        for point in points
    ])

inverse_transform = transform.GetInverseTransform()

nonterminal_moving_points_aligned = transform_points(
    nonterminal_moving_points,
    inverse_transform
)

terminal_moving_points_aligned = transform_points(
    terminal_moving_points,
    inverse_transform
)


print('nonterminal moving aligned: ', nonterminal_moving_points_aligned)
print('nonterminal fixed: ', nonterminal_fixed_points)

## Hungarian matching to create 1-1 branch point pairs
https://python.plainenglish.io/hungarian-algorithm-introduction-python-implementation-93e7c0890e15

Scipy gives us a one-liner

In [ ]:
max_distance = 30.

C = cdist(
    nonterminal_fixed_points,
    nonterminal_moving_points_aligned
)

rows, cols = linear_sum_assignment(C)
matched_distances = C[rows, cols]

# Keep only valid matches
valid = matched_distances <= max_distance

rows = rows[valid]
cols = cols[valid]
matched_distances = matched_distances[valid]

fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection="3d")

# Points
ax.scatter(
    nonterminal_fixed_points[:, 0],
    nonterminal_fixed_points[:, 1],
    nonterminal_fixed_points[:, 2],
    s=70,
    label="Fixed"
)

ax.scatter(
    nonterminal_moving_points_aligned[:, 0],
    nonterminal_moving_points_aligned[:, 1],
    nonterminal_moving_points_aligned[:, 2],
    s=70,
    label="Moving aligned"
)

# Only plot valid correspondence lines
for r, c, d in zip(rows, cols, matched_distances):

    p_fixed = nonterminal_fixed_points[r]
    p_moving = nonterminal_moving_points_aligned[c]

    ax.plot(
        [p_fixed[0], p_moving[0]],
        [p_fixed[1], p_moving[1]],
        [p_fixed[2], p_moving[2]],
        linewidth=1
    )

    midpoint = (p_fixed + p_moving) / 2

    ax.text(
        midpoint[0],
        midpoint[1],
        midpoint[2],
        f"{d:.1f}",
        fontsize=8
    )

# Label points
for i, point in enumerate(nonterminal_fixed_points):
    ax.text(
        point[0], point[1], point[2],
        f" F{i}",
        fontsize=8
    )

for i, point in enumerate(nonterminal_moving_points_aligned):
    ax.text(
        point[0], point[1], point[2],
        f" M{i}",
        fontsize=8
    )

ax.set_xlabel("X (mm)")
ax.set_ylabel("Y (mm)")
ax.set_zlabel("Z (mm)")
ax.set_title(f"Hungarian matches ≤ {max_distance} mm")
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
def physical_points_to_indices(points, image):
    return np.array([
        image.TransformPhysicalPointToContinuousIndex(tuple(point))
        for point in points
    ])


def show_branch_matches_on_coronal_mip(
    image,
    fixed_points,
    moving_points,
    rows,
    cols,
    distances,
    max_distance=5.0,
    alpha=0.7
):
    image_array = itk.array_view_from_image(image)

    mip = np.max(image_array, axis=1)

    fixed_indices = physical_points_to_indices(fixed_points, image)
    moving_indices = physical_points_to_indices(moving_points, image)

    fig, ax = plt.subplots(figsize=(12, 12))

    ax.imshow(
        mip,
        cmap="gray",
        interpolation="nearest",
        origin="upper"
    )

    for r, c, d in zip(rows, cols, distances):

        if d > max_distance:
            continue

        fixed = fixed_indices[r]
        moving = moving_indices[c]

        fx, fz = fixed[0], fixed[2]
        mx, mz = moving[0], moving[2]

        ax.plot(
            [fx, mx],
            [fz, mz],
            linewidth=1.5,
            alpha=alpha
        )

        ax.scatter(
            fx,
            fz,
            s=50,
            marker="o"
        )

        ax.scatter(
            mx,
            mz,
            s=50,
            marker="x"
        )

        ax.text(
            (fx + mx) / 2,
            (fz + mz) / 2,
            f"{d:.1f}",
            fontsize=8
        )

    ax.set_title(
        f"Branch-point correspondences — coronal MIP (≤ {max_distance} mm)"
    )
    ax.set_xlabel("X index")
    ax.set_ylabel("Z index")
    ax.set_aspect("equal")

    plt.tight_layout()
    plt.show()

show_branch_matches_on_coronal_mip(
    nonterminal_fixed,
    nonterminal_fixed_points,
    nonterminal_moving_points_aligned,
    rows,
    cols,
    matched_distances,
    max_distance=5.0
)

## Thin Plate Spline Deformation on Nonterminal Images

In [ ]:
TERMINAL_MATCH_MAX_COST_MM = 20.0  
ATTACHMENT_MIN_SPACING_MM = 2.0    


def indices_to_points(ijk, image):
    """physical points of (N, 3) continuous (i, j, k) array indices"""
    direction = itk.array_from_matrix(image.GetDirection())
    spacing = np.array(image.GetSpacing())
    start = np.array(image.GetLargestPossibleRegion().GetIndex())
    return np.array(image.GetOrigin()) + ((np.asarray(ijk, dtype=float) + start) * spacing) @ direction.T


def branch_centroids(label_image, branch_ids):
    array = itk.array_view_from_image(label_image)
    kji = ndimage.center_of_mass(array != 0, labels=array, index=[int(b) for b in branch_ids])
    return indices_to_points(np.array(kji)[:, ::-1], label_image)


# sibling branches will be almost the same point by definition so drop them
terminal_moving_ids = terminal_moving_points_df.branch_id.to_numpy()
terminal_fixed_ids = terminal_fixed_points_df.branch_id.to_numpy()

terminal_cost = (
    cdist(terminal_fixed_points, terminal_moving_points_aligned)
    + cdist(branch_centroids(terminal_fixed, terminal_fixed_ids),
            branch_centroids(terminal_moving, terminal_moving_ids))
)
fixed_rows, moving_cols = linear_sum_assignment(terminal_cost)
keep = terminal_cost[fixed_rows, moving_cols] <= TERMINAL_MATCH_MAX_COST_MM

# a pair should only be valid if their parent branches were matched
parent_of = dict(zip(branches_prediction.branch_id, branches_prediction.parent_id))
fixed_parent_of = dict(zip(branches_groundTruth.branch_id, branches_groundTruth.parent_id))
matched_nonterminal = {
    int(nonterminal_moving_points_df.branch_id.iloc[c]): int(nonterminal_fixed_points_df.branch_id.iloc[r])
    for r, c in zip(rows, cols)
}
same_parent = np.array([
    matched_nonterminal.get(int(parent_of[terminal_moving_ids[c]])) == int(fixed_parent_of[terminal_fixed_ids[r]])
    for r, c in zip(fixed_rows, moving_cols)
], dtype=bool)

print(f"terminal pairs dropped: {int((~keep).sum())} over the cost gate, "
      f"{int((keep & ~same_parent).sum())} whose parents weren't matched to each other")
fixed_rows, moving_cols = fixed_rows[keep & same_parent], moving_cols[keep & same_parent]

matched_fixed_branch = {
    int(terminal_moving_ids[c]): int(terminal_fixed_ids[r])
    for r, c in zip(fixed_rows, moving_cols)
}

pairs_by_parent = {}
for r, c in zip(fixed_rows, moving_cols):
    pairs_by_parent.setdefault(parent_of[terminal_moving_ids[c]], []).append((r, c))

attachment_source_points = np.array([
    terminal_moving_points_aligned[[c for _, c in pairs]].mean(0) for pairs in pairs_by_parent.values()
]).reshape(-1, 3)
attachment_target_points = np.array([
    terminal_fixed_points[[r for r, _ in pairs]].mean(0) for pairs in pairs_by_parent.values()
]).reshape(-1, 3)

if len(cols):
    far = cdist(attachment_source_points, nonterminal_moving_points_aligned[cols]).min(1) >= ATTACHMENT_MIN_SPACING_MM
    attachment_source_points = attachment_source_points[far]
    attachment_target_points = attachment_target_points[far]

print(f"matched {len(matched_fixed_branch)} of {len(terminal_moving_ids)} moving terminal branches "
      f"to {len(terminal_fixed_ids)} fixed ones")
print(f"adding {len(attachment_source_points)} final-bifurcation landmarks to the {len(cols)} nonterminal ones")

In [ ]:
source_points = nonterminal_moving_points_aligned[cols].astype(np.float64)
target_points = nonterminal_fixed_points[rows].astype(np.float64)

# plus one landmark per final bifurcation, where the terminal branches attach
source_points = np.vstack([source_points, attachment_source_points]).astype(np.float64)
target_points = np.vstack([target_points, attachment_target_points]).astype(np.float64)

In [ ]:
thin_plate_spline = itk.ThinPlateSplineKernelTransform[itk.D, Dimension].New()

source_landmarks = thin_plate_spline.GetSourceLandmarks()
source_landmarks.SetPoints(itk.vector_container_from_array(source_points.flatten()))
target_landmarks = thin_plate_spline.GetTargetLandmarks()
target_landmarks.SetPoints(itk.vector_container_from_array(target_points.flatten()))
thin_plate_spline.ComputeWMatrix()

deformed_nonterminal_moving_image = itk.resample_image_filter(
    nonterminal_moving,
    use_reference_image=True,
    reference_image=nonterminal_fixed,
    transform=thin_plate_spline.GetInverseTransform(),
    interpolator=itk.NearestNeighborInterpolateImageFunction.New(nonterminal_moving)
)

show_overlay(nonterminal_fixed, deformed_nonterminal_moving_image, labels=("nonterminal fixed", "nonterminal moving"))
show_overlay(nonterminal_fixed, nonterminal_moving, labels=("nonterminal fixed", "nonterminal moving"))

## Deformably Move Rigid Terminal Branches (must preserve branch geometry)

In [ ]:
TERMINAL_ROTATION_LIMIT_DEG = 15.0 # rotation limit
TERMINAL_ROTATION_STEP_DEG = 3.0    # grid step of the rotation search
TERMINAL_ROI_PADDING_MM = 5.0
GEOMETRY_TOLERANCE = 0.05           # allowed relative change in branch volume, length and width


def mask_points(mask):
    """physical points of a mask's foreground voxels"""
    return indices_to_points(np.argwhere(itk.array_view_from_image(mask) > 0.5)[:, ::-1], mask)


def branch_mask(label_image, branch_id):
    """float 0/1 image of one branch, on label_image's grid"""
    mask = itk.image_from_array((itk.array_view_from_image(label_image) == branch_id).astype(np.float32))
    mask.CopyInformation(label_image)
    return mask


def resample_mask(mask, transform, reference):
    """nearest-neighbour resample of a 0/1 mask onto reference's grid"""
    return itk.resample_image_filter(
        mask,
        transform=transform,
        interpolator=itk.NearestNeighborInterpolateImageFunction.New(mask),
        use_reference_image=True,
        reference_image=reference,
    )


def branch_box(masks, padding_mm):
    """(i, j, k) bounding box [lower, upper) of the masks' voxels, padded and clipped to the image"""
    kji = np.concatenate([np.argwhere(itk.array_view_from_image(m) > 0.5) for m in masks])
    pad = np.ceil(padding_mm / np.array(masks[0].GetSpacing())).astype(int)
    lower = np.maximum(kji.min(0)[::-1] - pad, 0)
    upper = np.minimum(kji.max(0)[::-1] + 1 + pad, np.array(itk.size(masks[0])))
    return lower, upper


def crop_to_box(image, lower, upper):
    """sub-image [lower, upper) in (i, j, k) array indices, at the same physical place"""
    array = itk.array_view_from_image(image)[lower[2]:upper[2], lower[1]:upper[1], lower[0]:upper[0]]
    cropped = itk.image_from_array(np.ascontiguousarray(array))
    cropped.SetSpacing(image.GetSpacing())
    cropped.SetDirection(image.GetDirection())
    cropped.SetOrigin(indices_to_points([lower], image)[0].tolist())
    return cropped


def dice(a, b):
    a = np.asarray(itk.array_view_from_image(a)) > 0.5
    b = np.asarray(itk.array_view_from_image(b)) > 0.5
    total = a.sum() + b.sum()
    return round(2.0 * (a & b).sum() / total, 3) if total else np.nan


def branch_geometry(mask):
    """volume (mm^3), length along the principal axis (mm) and mean width (mm)"""
    points = mask_points(mask)
    volume = len(points) * float(np.prod(mask.GetSpacing()))
    if len(points) < 2:
        return volume, 0.0, np.nan
    centered = points - points.mean(0)
    axis = np.linalg.svd(centered, full_matrices=False)[2][0]
    projection = centered @ axis
    length = projection.max() - projection.min()
    width = 2.0 * np.sqrt(volume / (np.pi * length)) if length > 0 else np.nan
    return volume, length, width


# allow a branch to rotate around the center within DOF
def bounded_rotation(fixed_roi, moving_mask, center):
    euler = itk.Euler3DTransform[itk.D].New()
    euler.SetCenter([float(v) for v in center])

    metric = itk.MeanSquaresImageToImageMetricv4[FloatImageType, FloatImageType].New()
    metric.SetFixedImage(fixed_roi)
    metric.SetMovingImage(moving_mask)
    metric.SetMovingTransform(euler)
    metric.Initialize()

    angles = np.deg2rad(np.arange(-TERMINAL_ROTATION_LIMIT_DEG,
                                  TERMINAL_ROTATION_LIMIT_DEG + 1e-9,
                                  TERMINAL_ROTATION_STEP_DEG))
    grid = np.array(np.meshgrid(angles, angles, angles, indexing="ij")).reshape(3, -1).T
    values = np.empty(len(grid))
    for n, (angle_x, angle_y, angle_z) in enumerate(grid):
        euler.SetRotation(float(angle_x), float(angle_y), float(angle_z))
        values[n] = metric.GetValue()

    near_best = np.flatnonzero(values <= values.min() + 0.01 * (values.max() - values.min()))
    best = near_best[np.argmin(np.linalg.norm(grid[near_best], axis=1))]
    return np.rad2deg(grid[best])


terminal_displacements = (
    transform_points(terminal_moving_points_aligned, thin_plate_spline) - terminal_moving_points_aligned
)

print(f"TPS shift of terminal branch points: median {np.median(np.linalg.norm(terminal_displacements, axis=1)):.2f} mm, "
      f"max {np.linalg.norm(terminal_displacements, axis=1).max():.2f} mm")

In [ ]:
terminal_alignment = []
terminal_branch_transforms = {}  
aligned_terminal_masks = {}

# for each branch of the terminal moving
for row, moving_branch in enumerate(terminal_moving_ids):
    moving_branch = int(moving_branch)
    fixed_branch = matched_fixed_branch.get(moving_branch)
    moving_mask = branch_mask(terminal_moving, moving_branch)
    branch_point = terminal_moving_points_aligned[row]
    displacement = terminal_displacements[row]

    # rigidly move via TPS displacement at connection point
    shift = itk.TranslationTransform[itk.D, Dimension].New()
    shift.SetOffset([float(-v) for v in displacement])
    shifted_mask = resample_mask(moving_mask, shift, fixed_image)
    rotation_center = branch_point + displacement

    rotation_deg = np.zeros(3)
    dice_trachea_aligned = dice_shifted = dice_rotated = np.nan
    if fixed_branch is not None:
        fixed_mask = branch_mask(terminal_fixed, fixed_branch)

        # crop to branch ROI
        reach = np.linalg.norm(mask_points(shifted_mask) - rotation_center, axis=1).max(initial=0.0)
        padding_mm = TERMINAL_ROI_PADDING_MM + reach * np.sin(np.deg2rad(TERMINAL_ROTATION_LIMIT_DEG))
        lower, upper = branch_box([shifted_mask, fixed_mask], padding_mm)
        fixed_roi = crop_to_box(fixed_mask, lower, upper)

        # limit rotation to 15 degrees in each direction to keep from locking to parent branch
        rotation_deg = bounded_rotation(fixed_roi, shifted_mask, rotation_center)

    # full rigid move
    branch_transform = itk.Euler3DTransform[itk.D].New()
    branch_transform.SetCenter([float(v) for v in rotation_center])
    branch_transform.SetRotation(*[float(v) for v in np.deg2rad(rotation_deg)])
    branch_transform.SetTranslation([float(-v) for v in displacement])
    aligned_mask = resample_mask(moving_mask, branch_transform, fixed_image)

    if fixed_branch is not None:
        dice_trachea_aligned = dice(moving_mask, fixed_mask)
        dice_shifted = dice(shifted_mask, fixed_mask)
        dice_rotated = dice(aligned_mask, fixed_mask)

    # branch length regularization
    before = branch_geometry(moving_mask)
    after = branch_geometry(aligned_mask)
    voxel_mm = min(fixed_image.GetSpacing())
    change, geometry_ok = {}, True
    for name, b, a, slack in zip(("volume", "length", "width"), before, after, (0.0, voxel_mm, voxel_mm)):
        change[name] = (a - b) / b if b else np.nan
        if np.isfinite(a) and np.isfinite(b):
            geometry_ok = geometry_ok and abs(a - b) <= GEOMETRY_TOLERANCE * b + slack

    terminal_branch_transforms[moving_branch] = branch_transform
    aligned_terminal_masks[moving_branch] = aligned_mask
    terminal_alignment.append({
        "moving_branch": moving_branch,
        "fixed_branch": fixed_branch,
        "shift_mm": round(float(np.linalg.norm(displacement)), 2),
        "rotation_deg": tuple(round(float(v), 1) for v in rotation_deg),
        "dice_trachea_aligned": dice_trachea_aligned,
        "dice_shifted": dice_shifted,
        "dice_rotated": dice_rotated,
        **{f"{name}_change": round(float(v), 3) for name, v in change.items()},
        "geometry_ok": geometry_ok,
    })

terminal_alignment = pd.DataFrame(terminal_alignment)

# Display alignment
final_aligned_array = itk.array_from_image(deformed_nonterminal_moving_image).copy()
terminal_overlap_voxels = 0
for moving_branch, aligned_mask in aligned_terminal_masks.items():
    inside = np.asarray(itk.array_view_from_image(aligned_mask)) > 0.5
    terminal_overlap_voxels += int((inside & (final_aligned_array != 0)).sum())
    final_aligned_array[inside & (final_aligned_array == 0)] = moving_branch

final_aligned_image = itk.image_from_array(final_aligned_array)
final_aligned_image.CopyInformation(deformed_nonterminal_moving_image)
terminal_moving_aligned = mask_to_values(final_aligned_image, list(aligned_terminal_masks))

print(f"rigidly moved {len(terminal_alignment)} terminal branches, rotated {int(terminal_alignment.fixed_branch.notna().sum())} "
      f"(the rest are shifted only: no fixed partner)")
print(f"geometry preserved (within {GEOMETRY_TOLERANCE:.0%}, plus a voxel for length and width): {int(terminal_alignment.geometry_ok.sum())} of {len(terminal_alignment)}")
print(f"terminal voxels that landed on the deformed nonterminal airway (kept nonterminal): {terminal_overlap_voxels}")

show_overlay(terminal_fixed, terminal_moving, labels=("terminal fixed", "terminal moving (trachea aligned)"))
show_overlay(terminal_fixed, terminal_moving_aligned, labels=("terminal fixed", "terminal moving (shifted + rotated)"))
show_overlay(fixed_image, final_aligned_image, labels=("fixed", "moving (final aligned)"))

terminal_alignment

## Displacement Demo

In [ ]:
ROTATION_DEMO_BRANCH = 198

rotated = terminal_alignment.dropna(subset=["fixed_branch"])
if rotated.empty:
    raise ValueError("no terminal branch was matched, so none was rotated")
if ROTATION_DEMO_BRANCH is None:
    ROTATION_DEMO_BRANCH = int(rotated.loc[rotated.rotation_deg.map(lambda a: np.abs(a).sum()).idxmax(), "moving_branch"])

demo = terminal_alignment.set_index("moving_branch").loc[ROTATION_DEMO_BRANCH]
demo_row = int(np.flatnonzero(terminal_moving_ids == ROTATION_DEMO_BRANCH)[0])
demo_rotation = np.array(demo.rotation_deg)
demo_shift = terminal_displacements[demo_row]
demo_point = terminal_moving_points_aligned[demo_row]
demo_center = demo_point + demo_shift

demo_moving = branch_mask(terminal_moving, ROTATION_DEMO_BRANCH)
demo_fixed = branch_mask(terminal_fixed, int(demo.fixed_branch))
lower, upper = branch_box([demo_moving, demo_fixed, aligned_terminal_masks[ROTATION_DEMO_BRANCH]], TERMINAL_ROI_PADDING_MM)
demo_roi = crop_to_box(demo_fixed, lower, upper)
roi = np.s_[lower[2]:upper[2], lower[1]:upper[1], lower[0]:upper[0]]
fixed_branch_roi = np.asarray(itk.array_view_from_image(demo_fixed))[roi] > 0.5
airway_roi = np.asarray(itk.array_view_from_image(fixed_image))[roi] != 0


def demo_transform(shift_fraction, rotation_fraction):
    """part of the branch's rigid move, fixed -> aligned moving (what resampling wants)"""
    transform = itk.Euler3DTransform[itk.D].New()
    transform.SetCenter([float(v) for v in demo_center])
    transform.SetRotation(*[float(v) for v in np.deg2rad(rotation_fraction * demo_rotation)])
    transform.SetTranslation([float(-v) for v in shift_fraction * demo_shift])
    return transform


steps = (
    [("trachea aligned", 0.0, 0.0)] * 3
    + [("shifting with the TPS", s, 0.0) for s in np.linspace(0, 1, 8)[1:]]
    + [("rotating about the branch point", 1.0, a) for a in np.linspace(0, 1, 12)[1:]]
    + [("in position", 1.0, 1.0)] * 4
)

frames = []
for label, s, a in steps:
    moving_roi = itk.array_from_image(resample_mask(demo_moving, demo_transform(s, a), demo_roi)) > 0.5
    overlap = 2 * (moving_roi & fixed_branch_roi).sum() / max(moving_roi.sum() + fixed_branch_roi.sum(), 1)
    frames.append((label, s, a, moving_roi, overlap))


def overlay_rgb(moving_roi, axis):
    """moving = red, fixed branch = blue, overlap = white, rest of the fixed airway = grey"""
    red = moving_roi.max(axis=axis).astype(float)
    blue = fixed_branch_roi.max(axis=axis).astype(float)
    grey = 0.25 * airway_roi.max(axis=axis)
    return np.stack([np.maximum(red, grey), np.maximum(np.minimum(red, blue), grey), np.maximum(blue, grey)], axis=-1)


# (title, projected numpy axis, (column, row) of (i, j, k)) -- same views as show_mip
views = [("Axial MIP", 0, (0, 1)), ("Coronal MIP", 1, (0, 2)), ("Sagittal MIP", 2, (1, 2))]
roi_spacing = np.array(demo_roi.GetSpacing())

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
images, markers = [], []
for ax, (title, axis, (col, row)) in zip(axes, views):
    images.append(ax.imshow(overlay_rgb(frames[0][3], axis), aspect=roi_spacing[row] / roi_spacing[col]))
    markers.append(ax.plot([], [], "+", color="yellow", markersize=14, markeredgewidth=2)[0])
    ax.set_title(title)
    ax.axis("off")
heading = fig.suptitle("")


def draw(n):
    label, s, a, moving_roi, overlap = frames[n]
    point = physical_points_to_indices([demo_point + s * demo_shift], demo_roi)[0]
    for image, marker, (_, axis, (col, row)) in zip(images, markers, views):
        image.set_data(overlay_rgb(moving_roi, axis))
        marker.set_data([point[col]], [point[row]])
    angles = ", ".join(f"{v:+.0f}" for v in a * demo_rotation)
    heading.set_text(
        f"branch {ROTATION_DEMO_BRANCH} -> fixed {int(demo.fixed_branch)}: {label}   |   "
        f"shift {s * np.linalg.norm(demo_shift):.1f} mm, rotation ({angles}) deg, Dice {overlap:.2f}\n"
        "moving = red, fixed branch = blue, overlap = white, rest of fixed airway = grey, + = branch point"
    )
    return images + markers + [heading]


fig.tight_layout(rect=(0, 0, 1, 0.86))  # leave room for the two-line heading
rotation_animation = animation.FuncAnimation(fig, draw, frames=len(frames), interval=200)
plt.close(fig)
HTML(rotation_animation.to_jshtml())

## Write Registered Image

In [ ]:
# binarize masks
final_mask = itk.image_from_array(
    (itk.array_view_from_image(final_aligned_image) != 0).astype(np.uint8)
)
final_mask.CopyInformation(final_aligned_image)

registered_image = itk.resample_image_filter(
    final_mask,
    interpolator=itk.NearestNeighborInterpolateImageFunction.New(final_mask),
    use_reference_image=True,
    reference_image=fixed_image,
)

itk.imwrite(registered_image, "warpedMoving.nii.gz")

final_fixed = itk.image_from_array(
    (itk.array_view_from_image(fixed_image) != 0).astype(np.uint8)
)
final_fixed.CopyInformation(fixed_image)
itk.imwrite(final_fixed, "fixed.nii.gz")

## Rigid Step from Centerlines Instead of Mask MSE

The per-branch rigid step above scores a candidate rotation with `MeanSquaresImageToImageMetricv4`
on the binary branch masks, so the score is driven by voxel overlap: a thick proximal branch
outweighs a thin distal one, and a rotation that fattens the intersection scores as well as one
that actually follows the branch axis.

Here the same bounded search is scored on the labeled *skeletons* instead -- the symmetric mean
nearest-neighbour distance (in mm) between the moving branch centerline and its matched fixed
branch centerline. Everything else is held fixed: same TPS displacement, same rotation center,
same +/-15 deg limit and grid step, same tie-break toward the smallest rotation. Only the cost
changes, so the two arms are directly comparable, and the cost is now in millimetres rather than
in units of mask overlap.

In [ ]:
def branch_centerline_points(skeleton_labels_image, branch_id):
    """physical points of one branch's labeled skeleton voxels"""
    kji = np.argwhere(np.asarray(itk.array_view_from_image(skeleton_labels_image)) == int(branch_id))
    return indices_to_points(kji[:, ::-1], skeleton_labels_image).reshape(-1, 3)


# the labeled skeletons sit in each image's own frame, so the moving centerlines need the same
# trachea alignment the moving masks already got
moving_centerlines = {}
for branch_id in terminal_moving_ids:
    points = branch_centerline_points(branchLabelsImagePrediction, branch_id)
    moving_centerlines[int(branch_id)] = (
        transform_points(points, inverse_transform) if len(points) else points
    )

fixed_centerlines = {
    int(branch_id): branch_centerline_points(branchLabelsImageGroundTruth, branch_id)
    for branch_id in terminal_fixed_ids
}

moving_sizes = np.array([len(p) for p in moving_centerlines.values()])
fixed_sizes = np.array([len(p) for p in fixed_centerlines.values()])

print(f"moving terminal centerlines: {len(moving_centerlines)}, median {np.median(moving_sizes):.0f} voxels, "
      f"{int((moving_sizes < 2).sum())} with fewer than 2")
print(f" fixed terminal centerlines: {len(fixed_centerlines)}, median {np.median(fixed_sizes):.0f} voxels, "
      f"{int((fixed_sizes < 2).sum())} with fewer than 2")

In [ ]:
CENTERLINE_TIE_TOLERANCE_MM = 0.05  # rotations scoring this close to the best one count as ties

_matrix_transform = itk.Euler3DTransform[itk.D].New()


def rotation_matrix(rotation_rad):
    """ITK's Euler3D rotation matrix, so the angles mean what they mean to the branch transforms"""
    _matrix_transform.SetIdentity()
    _matrix_transform.SetRotation(*[float(v) for v in rotation_rad])
    return itk.array_from_matrix(_matrix_transform.GetMatrix())


def moved_by(points, matrix, center, translation):
    """where points land when their mask is resampled with this Euler transform

    resample_image_filter maps output points back to input points, so the mask -- and the
    centerline riding along with it -- moves by the transform's inverse.
    """
    center = np.asarray(center, dtype=float)
    offset = center + np.asarray(translation, dtype=float) - matrix @ center
    return (np.asarray(points, dtype=float) - offset) @ matrix


def moved_points(points, rotation_deg, center, translation):
    return moved_by(points, rotation_matrix(np.deg2rad(rotation_deg)), center, translation)


def centerline_error(moving_centerline, fixed_centerline):
    """symmetric mean nearest-neighbour distance between two centerlines, in mm"""
    if len(moving_centerline) == 0 or len(fixed_centerline) == 0:
        return np.nan
    distance = cdist(moving_centerline, fixed_centerline)
    return float(0.5 * (distance.min(1).mean() + distance.min(0).mean()))


_angles = np.deg2rad(np.arange(-TERMINAL_ROTATION_LIMIT_DEG,
                               TERMINAL_ROTATION_LIMIT_DEG + 1e-9,
                               TERMINAL_ROTATION_STEP_DEG))
ROTATION_GRID = np.array(np.meshgrid(_angles, _angles, _angles, indexing="ij")).reshape(3, -1).T
ROTATION_MATRICES = [rotation_matrix(angles) for angles in ROTATION_GRID]


def bounded_rotation_centerline(moving_centerline, fixed_centerline, center, translation):
    """bounded_rotation, scored on centerline distance instead of mask mean squares"""
    values = np.array([
        centerline_error(moved_by(moving_centerline, matrix, center, translation), fixed_centerline)
        for matrix in ROTATION_MATRICES
    ])

    # ties within a twentieth of a millimetre (or 1% of the range) go to the smallest rotation
    tolerance = max(CENTERLINE_TIE_TOLERANCE_MM, 0.01 * (values.max() - values.min()))
    near_best = np.flatnonzero(values <= values.min() + tolerance)
    best = near_best[np.argmin(np.linalg.norm(ROTATION_GRID[near_best], axis=1))]
    return np.rad2deg(ROTATION_GRID[best])

In [ ]:
def geometry_change(before_mask, after_mask):
    """relative change in volume, length and width, and whether it stays inside the tolerance"""
    before, after = branch_geometry(before_mask), branch_geometry(after_mask)
    voxel_mm = min(fixed_image.GetSpacing())
    change, ok = {}, True
    for name, b, a, slack in zip(("volume", "length", "width"), before, after, (0.0, voxel_mm, voxel_mm)):
        change[name] = (a - b) / b if b else np.nan
        if np.isfinite(a) and np.isfinite(b):
            ok = ok and abs(a - b) <= GEOMETRY_TOLERANCE * b + slack
    return change, ok


# the rotations the mask MSE search picked, to score on the same ruler
mse_rotation_of = {
    int(branch): np.array(rotation, dtype=float)
    for branch, rotation in zip(terminal_alignment.moving_branch, terminal_alignment.rotation_deg)
}

centerline_alignment = []
centerline_branch_transforms = {}
centerline_aligned_masks = {}

for row, moving_branch in enumerate(terminal_moving_ids):
    moving_branch = int(moving_branch)
    fixed_branch = matched_fixed_branch.get(moving_branch)
    moving_mask = branch_mask(terminal_moving, moving_branch)
    branch_point = terminal_moving_points_aligned[row]
    displacement = terminal_displacements[row]
    translation = -displacement                    # so the branch moves by +displacement
    rotation_center = branch_point + displacement

    moving_centerline = moving_centerlines[moving_branch]
    fixed_centerline = (fixed_centerlines[fixed_branch] if fixed_branch is not None
                        else np.empty((0, 3)))

    # no partner, or nothing to score against -> shift only, as before
    searchable = len(moving_centerline) > 0 and len(fixed_centerline) > 0
    rotation_deg = (
        bounded_rotation_centerline(moving_centerline, fixed_centerline, rotation_center, translation)
        if searchable else np.zeros(3)
    )

    branch_transform = itk.Euler3DTransform[itk.D].New()
    branch_transform.SetCenter([float(v) for v in rotation_center])
    branch_transform.SetRotation(*[float(v) for v in np.deg2rad(rotation_deg)])
    branch_transform.SetTranslation([float(v) for v in translation])
    aligned_mask = resample_mask(moving_mask, branch_transform, fixed_image)

    def error_at(angles_deg):
        return centerline_error(
            moved_points(moving_centerline, angles_deg, rotation_center, translation),
            fixed_centerline,
        )

    dice_rotated = np.nan
    if fixed_branch is not None:
        dice_rotated = dice(aligned_mask, branch_mask(terminal_fixed, fixed_branch))

    change, geometry_ok = geometry_change(moving_mask, aligned_mask)

    centerline_branch_transforms[moving_branch] = branch_transform
    centerline_aligned_masks[moving_branch] = aligned_mask
    centerline_alignment.append({
        "moving_branch": moving_branch,
        "fixed_branch": fixed_branch,
        "shift_mm": round(float(np.linalg.norm(displacement)), 2),
        "rotation_deg": tuple(round(float(v), 1) for v in rotation_deg),
        "error_trachea_aligned_mm": centerline_error(moving_centerline, fixed_centerline),
        "error_shifted_mm": error_at(np.zeros(3)),
        "error_mse_rotation_mm": error_at(mse_rotation_of.get(moving_branch, np.zeros(3))),
        "error_centerline_rotation_mm": error_at(rotation_deg),
        "dice_rotated": dice_rotated,
        **{f"{name}_change": round(float(v), 3) for name, v in change.items()},
        "geometry_ok": geometry_ok,
    })

error_columns = ["error_trachea_aligned_mm", "error_shifted_mm",
                 "error_mse_rotation_mm", "error_centerline_rotation_mm"]
centerline_alignment = pd.DataFrame(centerline_alignment).round({c: 2 for c in error_columns})

# same compositing as the mask MSE arm: deformed nonterminal airway, terminal branches dropped in
final_aligned_array_centerline = itk.array_from_image(deformed_nonterminal_moving_image).copy()
terminal_overlap_voxels_centerline = 0
for moving_branch, aligned_mask in centerline_aligned_masks.items():
    inside = np.asarray(itk.array_view_from_image(aligned_mask)) > 0.5
    terminal_overlap_voxels_centerline += int((inside & (final_aligned_array_centerline != 0)).sum())
    final_aligned_array_centerline[inside & (final_aligned_array_centerline == 0)] = moving_branch

final_aligned_image_centerline = itk.image_from_array(final_aligned_array_centerline)
final_aligned_image_centerline.CopyInformation(deformed_nonterminal_moving_image)
terminal_moving_aligned_centerline = mask_to_values(final_aligned_image_centerline,
                                                    list(centerline_aligned_masks))

rotated = int(centerline_alignment.fixed_branch.notna().sum())
print(f"rigidly moved {len(centerline_alignment)} terminal branches, rotated {rotated} on centerline distance "
      f"(the rest are shifted only: no fixed partner)")
print(f"geometry preserved (within {GEOMETRY_TOLERANCE:.0%}, plus a voxel for length and width): "
      f"{int(centerline_alignment.geometry_ok.sum())} of {len(centerline_alignment)}")
print(f"terminal voxels that landed on the deformed nonterminal airway (kept nonterminal): "
      f"{terminal_overlap_voxels_centerline}")

centerline_alignment

In [ ]:
matched = centerline_alignment.dropna(subset=["fixed_branch"])


def error_summary(label, errors):
    errors = np.asarray(errors, dtype=float)
    print(f"{label:>30}: median {np.median(errors):5.2f}   mean {errors.mean():5.2f}   "
          f"95th {np.percentile(errors, 95):5.2f}   max {errors.max():5.2f}")


print(f"centerline distance to the matched fixed branch, mm ({len(matched)} matched terminal branches)")
error_summary("trachea aligned only", matched.error_trachea_aligned_mm)
error_summary("+ TPS shift", matched.error_shifted_mm)
error_summary("+ rotation from mask MSE", matched.error_mse_rotation_mm)
error_summary("+ rotation from centerlines", matched.error_centerline_rotation_mm)

improvement = matched.error_mse_rotation_mm - matched.error_centerline_rotation_mm
print(f"\nvs the mask MSE rotation: better on {int((improvement > 1e-6).sum())} branches, "
      f"tied on {int((improvement.abs() <= 1e-6).sum())}, worse on {int((improvement < -1e-6).sum())}")
print(f"median change {improvement.median():+.2f} mm, worst regression {improvement.min():+.2f} mm")

magnitude = lambda column: np.linalg.norm(np.array(column.tolist(), dtype=float), axis=1)
mse_matched = terminal_alignment.set_index("moving_branch").loc[matched.moving_branch]
print(f"\nrotation magnitude: mask MSE median {np.median(magnitude(mse_matched.rotation_deg)):.1f} deg, "
      f"centerline median {np.median(magnitude(matched.rotation_deg)):.1f} deg")
print(f"per-branch Dice: mask MSE median {mse_matched.dice_rotated.median():.3f}, "
      f"centerline median {matched.dice_rotated.median():.3f}")

comparison = matched.set_index("moving_branch")[
    ["fixed_branch", "shift_mm", "rotation_deg", "error_shifted_mm",
     "error_mse_rotation_mm", "error_centerline_rotation_mm", "dice_rotated", "geometry_ok"]
].join(
    mse_matched[["rotation_deg", "dice_rotated"]].add_suffix("_mse")
).assign(improvement_mm=improvement.to_numpy().round(2))

show_overlay(terminal_fixed, terminal_moving_aligned,
             labels=("terminal fixed", "terminal moving (mask MSE rotation)"))
show_overlay(terminal_fixed, terminal_moving_aligned_centerline,
             labels=("terminal fixed", "terminal moving (centerline rotation)"))
show_overlay(fixed_image, final_aligned_image_centerline,
             labels=("fixed", "moving (final aligned, centerline rotation)"))

# to write this arm out with the cell above:
# final_aligned_image = final_aligned_image_centerline

comparison.sort_values("error_centerline_rotation_mm", ascending=False).head(20)

## Does the Centerline Cost Actually Do Better?

Both arms see the same branches, the same TPS shift and the same rotation center, and differ only
in how a candidate rotation is scored -- so the honest statistic is the **paired per-branch
difference**, not two independent distributions. Positive means the centerline rotation left that
branch closer to its fixed partner than the mask MSE rotation did.

Two anchors keep the numbers interpretable:

- a difference below the voxel size is not a real improvement, however consistent its sign
- branches where both searches picked the *same* rotation are ties by construction, and carrying
  them in the summary dilutes whatever real effect exists

In [ ]:
from scipy.stats import wilcoxon

# light-mode slots from the validated palette: two categorical hues for the arms,
# a blue/red diverging pair for win/loss polarity, gray for context
PALETTE = {
    "mse": "#2a78d6",
    "centerline": "#eb6834",
    "better": "#2a78d6",
    "worse": "#e34948",
    "tie": "#b9b8b3",
    "context": "#8a8984",
    "baseline": "#52514e",
    "grid": "#e8e7e2",
    "axis": "#d6d5d0",
    "surface": "#fcfcfb",
    "ink": "#0b0b0b",
    "muted": "#52514e",
}

TIE_MM = 0.01          # below this the two arms agree to rounding
VOXEL_MM = float(min(fixed_image.GetSpacing()))


def bootstrap_ci(values, statistic=np.median, draws=10000, seed=0):
    """percentile bootstrap interval for a paired statistic"""
    values = np.asarray(values, dtype=float)
    if len(values) < 2:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    resampled = statistic(rng.choice(values, size=(draws, len(values)), replace=True), axis=1)
    return tuple(np.percentile(resampled, [2.5, 97.5]))


paired = (
    centerline_alignment.dropna(subset=["fixed_branch"]).set_index("moving_branch")
    .join(terminal_alignment.set_index("moving_branch")[["rotation_deg", "dice_rotated"]]
          .add_suffix("_mse"))
    .join(branches_prediction.set_index("branch_id")[["generation"]])
)

paired["improvement_mm"] = paired.error_mse_rotation_mm - paired.error_centerline_rotation_mm
paired["improvement_pct"] = 100.0 * paired.improvement_mm / paired.error_mse_rotation_mm.replace(0.0, np.nan)
paired["dice_delta"] = paired.dice_rotated - paired.dice_rotated_mse
paired["same_rotation"] = [
    tuple(np.round(a, 1)) == tuple(np.round(b, 1))
    for a, b in zip(paired.rotation_deg, paired.rotation_deg_mse)
]

improvement = paired.improvement_mm.to_numpy()
disagreed = paired[~paired.same_rotation]

wins = int((improvement > TIE_MM).sum())
losses = int((improvement < -TIE_MM).sum())
ties = len(paired) - wins - losses

print(f"{len(paired)} matched terminal branches, voxel {VOXEL_MM:.2f} mm")
print(f"the two searches picked the same rotation on {int(paired.same_rotation.sum())}, "
      f"disagreed on {len(disagreed)}")
print(f"centerline closer on {wins}, tied on {ties}, farther on {losses}\n")

median_ci = bootstrap_ci(improvement)
mean_ci = bootstrap_ci(improvement, statistic=np.mean)
print(f"paired error reduction over all {len(paired)} branches (mm, + = centerline closer)")
print(f"  median {np.median(improvement):+.3f}  95% CI [{median_ci[0]:+.3f}, {median_ci[1]:+.3f}]")
print(f"    mean {improvement.mean():+.3f}  95% CI [{mean_ci[0]:+.3f}, {mean_ci[1]:+.3f}]")
print(f"   range {improvement.min():+.3f} to {improvement.max():+.3f}")

if len(disagreed):
    disagreed_improvement = disagreed.improvement_mm.to_numpy()
    disagreed_ci = bootstrap_ci(disagreed_improvement)
    print(f"\nover the {len(disagreed)} branches where the rotations differ")
    print(f"  median {np.median(disagreed_improvement):+.3f}  "
          f"95% CI [{disagreed_ci[0]:+.3f}, {disagreed_ci[1]:+.3f}]")
    print(f"  median relative {disagreed.improvement_pct.median():+.1f}% of the mask MSE error")

nonzero = improvement[np.abs(improvement) > TIE_MM]
if len(nonzero) >= 6:
    print(f"\nWilcoxon signed-rank on the {len(nonzero)} non-tied branches: "
          f"p = {wilcoxon(nonzero).pvalue:.4f}")
else:
    print(f"\ntoo few non-tied branches ({len(nonzero)}) for a signed-rank test")

low, high = median_ci
if low > VOXEL_MM:
    verdict = "centerline is better by more than a voxel"
elif low > 0:
    verdict = f"centerline is consistently better, but by less than a voxel ({VOXEL_MM:.2f} mm)"
elif high < 0:
    verdict = "mask MSE is better"
else:
    verdict = "no separation: the interval spans zero"
print(f"\nverdict: {verdict}")

print(f"\nper-branch Dice change (secondary): median {paired.dice_delta.median():+.3f}, "
      f"better on {int((paired.dice_delta > 0).sum())}, worse on {int((paired.dice_delta < 0).sum())}")

stage_summary = pd.DataFrame({
    stage: {
        "median": paired[column].median(),
        "mean": paired[column].mean(),
        "p95": paired[column].quantile(0.95),
        "max": paired[column].max(),
    }
    for stage, column in [
        ("trachea aligned", "error_trachea_aligned_mm"),
        ("+ TPS shift", "error_shifted_mm"),
        ("+ mask MSE rotation", "error_mse_rotation_mm"),
        ("+ centerline rotation", "error_centerline_rotation_mm"),
    ]
}).T.round(3)
stage_summary

In [ ]:
from matplotlib.lines import Line2D


def style_axes(ax, xlabel="", ylabel="", title=""):
    ax.set_facecolor(PALETTE["surface"])
    ax.grid(True, color=PALETTE["grid"], linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(PALETTE["axis"])
    ax.tick_params(colors=PALETTE["muted"], labelsize=9)
    ax.set_xlabel(xlabel, color=PALETTE["muted"], fontsize=10)
    ax.set_ylabel(ylabel, color=PALETTE["muted"], fontsize=10)
    ax.set_title(title, color=PALETTE["ink"], fontsize=11, loc="left", pad=8)


def polarity_colors(values):
    """diverging blue / gray / red by the sign of a per-branch difference"""
    return np.where(np.asarray(values) > TIE_MM, PALETTE["better"],
                    np.where(np.asarray(values) < -TIE_MM, PALETTE["worse"], PALETTE["tie"]))


def ecdf(values):
    values = np.sort(np.asarray(values, dtype=float))
    return values, np.arange(1, len(values) + 1) / len(values)


fig, axes = plt.subplots(2, 2, figsize=(14, 10), facecolor=PALETTE["surface"])
fig.suptitle(
    f"Centerline vs mask MSE rotation, per terminal branch (n = {len(paired)})",
    color=PALETTE["ink"], fontsize=13, x=0.02, ha="left",
)

# (a) paired scatter: below the diagonal, the centerline rotation left the branch closer
ax = axes[0, 0]
limit = 1.08 * max(paired.error_mse_rotation_mm.max(), paired.error_centerline_rotation_mm.max())
ax.plot([0, limit], [0, limit], color=PALETTE["context"], linewidth=1.5, linestyle="--", zorder=1)
ax.scatter(paired.error_mse_rotation_mm, paired.error_centerline_rotation_mm,
           s=60, c=polarity_colors(paired.improvement_mm),
           edgecolor=PALETTE["surface"], linewidth=1.0, zorder=3)
for branch, row in paired.reindex(paired.improvement_mm.abs().sort_values(ascending=False).index).head(3).iterrows():
    ax.annotate(f" {branch}", (row.error_mse_rotation_mm, row.error_centerline_rotation_mm),
                color=PALETTE["muted"], fontsize=9, va="center")
ax.set_xlim(0, limit)
ax.set_ylim(0, limit)
ax.set_aspect("equal")
style_axes(ax, "error after mask MSE rotation (mm)", "error after centerline rotation (mm)",
           "Below the dashed line = centerline closer")
ax.legend(handles=[
    Line2D([0], [0], marker="o", linestyle="", markersize=8, markerfacecolor=PALETTE["better"],
           markeredgecolor=PALETTE["surface"], label="centerline closer"),
    Line2D([0], [0], marker="o", linestyle="", markersize=8, markerfacecolor=PALETTE["tie"],
           markeredgecolor=PALETTE["surface"], label="tied"),
    Line2D([0], [0], marker="o", linestyle="", markersize=8, markerfacecolor=PALETTE["worse"],
           markeredgecolor=PALETTE["surface"], label="mask MSE closer"),
], frameon=False, fontsize=9, labelcolor=PALETTE["muted"], loc="lower right")

# (b) the paired differences themselves, sorted -- the headline "how much better"
ax = axes[0, 1]
sorted_improvement = paired.improvement_mm.sort_values(ascending=False)
ax.bar(np.arange(len(sorted_improvement)), sorted_improvement.to_numpy(),
       color=polarity_colors(sorted_improvement), width=0.85)
ax.axhline(0, color=PALETTE["ink"], linewidth=1)
for sign in (1, -1):
    ax.axhline(sign * VOXEL_MM, color=PALETTE["context"], linewidth=1, linestyle=":")
ax.annotate(f"+/- one voxel ({VOXEL_MM:.2f} mm)", (len(sorted_improvement) - 1, VOXEL_MM),
            color=PALETTE["muted"], fontsize=9, ha="right", va="bottom")
style_axes(ax, "terminal branches, sorted", "error reduction (mm)",
           f"Median {np.median(improvement):+.3f} mm, 95% CI [{median_ci[0]:+.3f}, {median_ci[1]:+.3f}]")

# (c) where the whole distribution sits after each stage
ax = axes[1, 0]
stages = [
    ("trachea aligned", paired.error_trachea_aligned_mm, PALETTE["context"], 1.5, ":"),
    ("+ TPS shift", paired.error_shifted_mm, PALETTE["baseline"], 1.5, "--"),
    ("+ mask MSE rotation", paired.error_mse_rotation_mm, PALETTE["mse"], 2.0, "-"),
    ("+ centerline rotation", paired.error_centerline_rotation_mm, PALETTE["centerline"], 2.0, "-"),
]
for label, values, color, width, dash in stages:
    x, y = ecdf(values)
    ax.step(x, y, where="post", color=color, linewidth=width, linestyle=dash, label=label)
ax.axvline(VOXEL_MM, color=PALETTE["context"], linewidth=1, linestyle=":")
ax.annotate(" one voxel", (VOXEL_MM, 0.04), color=PALETTE["muted"], fontsize=9)
ax.set_ylim(0, 1.02)
style_axes(ax, "centerline distance to the fixed partner (mm)", "fraction of branches at or below",
           "The two arms' curves separate only if the cost matters")
ax.legend(frameon=False, fontsize=9, labelcolor=PALETTE["muted"], loc="lower right")

# (d) does it help where the airway is small?
ax = axes[1, 1]
rng = np.random.default_rng(0)
generations = paired.generation.to_numpy()
jitter = rng.uniform(-0.18, 0.18, len(paired))
ax.scatter(generations + jitter, paired.improvement_mm,
           s=55, c=polarity_colors(paired.improvement_mm),
           edgecolor=PALETTE["surface"], linewidth=1.0, zorder=3)
for generation, group in paired.groupby("generation"):
    ax.plot([generation - 0.32, generation + 0.32], [group.improvement_mm.median()] * 2,
            color=PALETTE["ink"], linewidth=2, zorder=4)
ax.axhline(0, color=PALETTE["ink"], linewidth=1)
ax.set_xticks(sorted(paired.generation.unique()))
style_axes(ax, "generation", "error reduction (mm)",
           "Per branch, with the generation median in black")

fig.tight_layout(rect=(0, 0, 1, 0.95))
plt.show()

In [ ]:
from matplotlib.colors import ListedColormap, LinearSegmentedColormap

# sequential blue ramp (palette steps 100 -> 700) for the residual, gray silhouette for context
BLUE_RAMP = LinearSegmentedColormap.from_list(
    "airway_blue", ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"])
SILHOUETTE = ListedColormap([PALETTE["surface"], "#dcdbd6"])

fixed_origins = branches_groundTruth.set_index("branch_id")[["origin_x", "origin_y", "origin_z"]]
branch_points = fixed_origins.loc[paired.fixed_branch.astype(int)].to_numpy()
branch_indices = physical_points_to_indices(branch_points, fixed_image)
column, row = branch_indices[:, 0], branch_indices[:, 2]        # coronal MIP: x across, z down
silhouette = (np.asarray(itk.array_view_from_image(fixed_image)) != 0).max(axis=1).astype(np.uint8)

fig, axes = plt.subplots(1, 2, figsize=(15, 9), facecolor=PALETTE["surface"])

# left: which branches the centerline cost actually moved closer, and by how much
ax = axes[0]
ax.imshow(silhouette, cmap=SILHOUETTE, vmin=0, vmax=1, interpolation="nearest")
magnitude = paired.improvement_mm.abs().to_numpy()
scale = magnitude.max() if magnitude.max() > 0 else 1.0
ax.scatter(column, row, s=30 + 320 * magnitude / scale,
           c=polarity_colors(paired.improvement_mm),
           edgecolor=PALETTE["surface"], linewidth=1.2, zorder=3)
biggest = paired.reindex(paired.improvement_mm.abs().sort_values(ascending=False).index).head(5)
for branch in biggest.index:
    at = np.flatnonzero(paired.index == branch)[0]
    ax.annotate(f"  {branch}: {paired.improvement_mm.loc[branch]:+.2f} mm",
                (column[at], row[at]), color=PALETTE["ink"], fontsize=9, va="center")
ax.set_title("Error reduction from the centerline cost\n(dot area = size of the change)",
             color=PALETTE["ink"], fontsize=11, loc="left")
ax.axis("off")
ax.legend(handles=[
    Line2D([0], [0], marker="o", linestyle="", markersize=9, markerfacecolor=PALETTE["better"],
           markeredgecolor=PALETTE["surface"], label="centerline closer"),
    Line2D([0], [0], marker="o", linestyle="", markersize=9, markerfacecolor=PALETTE["tie"],
           markeredgecolor=PALETTE["surface"], label="tied"),
    Line2D([0], [0], marker="o", linestyle="", markersize=9, markerfacecolor=PALETTE["worse"],
           markeredgecolor=PALETTE["surface"], label="mask MSE closer"),
], frameon=False, fontsize=9, labelcolor=PALETTE["muted"], loc="lower right")

# right: what is left over after the centerline rotation, wherever it is left over
ax = axes[1]
ax.imshow(silhouette, cmap=SILHOUETTE, vmin=0, vmax=1, interpolation="nearest")
residual = paired.error_centerline_rotation_mm.to_numpy()
marks = ax.scatter(column, row, s=90, c=residual, cmap=BLUE_RAMP,
                   vmin=0, vmax=max(residual.max(), VOXEL_MM),
                   edgecolor=PALETTE["surface"], linewidth=1.2, zorder=3)
worst = paired.error_centerline_rotation_mm.sort_values(ascending=False).head(5)
for branch in worst.index:
    at = np.flatnonzero(paired.index == branch)[0]
    ax.annotate(f"  {branch}: {worst.loc[branch]:.2f} mm", (column[at], row[at]),
                color=PALETTE["ink"], fontsize=9, va="center")
ax.set_title("Centerline distance remaining after the centerline rotation",
             color=PALETTE["ink"], fontsize=11, loc="left")
ax.axis("off")
bar = fig.colorbar(marks, ax=ax, fraction=0.035, pad=0.02)
bar.set_label("mm to the fixed partner", color=PALETTE["muted"], fontsize=10)
bar.ax.tick_params(colors=PALETTE["muted"], labelsize=9)
bar.outline.set_visible(False)

fig.tight_layout()
plt.show()

# the branches worth looking at by hand: biggest gains, then biggest regressions
paired.sort_values("improvement_mm", ascending=False)[
    ["fixed_branch", "generation", "shift_mm", "rotation_deg", "rotation_deg_mse",
     "error_mse_rotation_mm", "error_centerline_rotation_mm", "improvement_mm",
     "dice_rotated", "dice_rotated_mse", "dice_delta"]
].head(10)